# JEDI-7B grounding smoke test

Goal: prove JEDI-7B loads, responds to a click query on a DesignBench screenshot, and returns coords that rescale to the right spot on the original image.

**Prereqs:** Colab Pro (A100 priority), HF read token, Google Colab VS Code extension.

Run cells top-to-bottom. If cell 6's red dot lands on the target element, JEDI is go for Day 1. See cell 7 for what worked, what fails silently, and how to reuse.

## Cell 1 — Bootstrap: Drive mount, repo clone, deps

In [ ]:
print("hello world")

In [ ]:
import os, subprocess, sys
from google.colab import drive

drive.mount('/content/drive', force_remount=False)

REPO_URL = 'https://github.com/isaacau502/GUI-grounded-gen'
REPO_DIR = '/content/GUI-grounded-gen'

if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', REPO_URL, REPO_DIR], check=True)
    head = subprocess.check_output(
        ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    print(f'Cloned fresh @ {head}')
else:
    before = subprocess.check_output(
        ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    subprocess.run(['git', '-C', REPO_DIR, 'pull'], check=True)
    after = subprocess.check_output(
        ['git', '-C', REPO_DIR, 'rev-parse', '--short', 'HEAD']).decode().strip()
    if before == after:
        print(f'No new commits (HEAD: {after})')
    else:
        print(f'Pulled {before} -> {after}:')
        log = subprocess.check_output(
            ['git', '-C', REPO_DIR, 'log', '--oneline', f'{before}..{after}']
        ).decode().strip()
        print(log)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'vllm', 'qwen-vl-utils', 'huggingface_hub', 'hf_transfer',
                'matplotlib', 'pillow'], check=True)

print('Bootstrap complete.')

## Cell 2 — HF token

VS Code Colab kernel can't pop the permission dialog that `userdata.get()` needs. Prompt via `getpass` instead — paste your HF read token when it asks. Runs once per session.

In [ ]:
import os, getpass
from huggingface_hub import HfApi

# VS Code Colab kernel can't access userdata.get() (no UI for permission prompt).
# Use getpass — prompt stays in session env, never hits notebook outputs.
if not os.environ.get('HF_TOKEN'):
    os.environ['HF_TOKEN'] = getpass.getpass('HF token: ')

me = HfApi().whoami()
print(f"HF authed as: {me.get('name', '?')}")

## Cell 3 — Weight download (JEDI + OmniParser fallback)

Two-step for speed: download to `/content/` (local SSD, ~100 MB/s from HF) then copy to Drive (persists across disconnects). `hf_transfer` enabled for xet fast path. Idempotent — re-run skips if already in Drive.

In [ ]:
import os, shutil
from huggingface_hub import snapshot_download

os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

JEDI_LOCAL = '/content/jedi-weights'
JEDI_DRIVE = '/content/drive/MyDrive/jedi-weights'
OMNI_LOCAL = '/content/omniparser-weights'
OMNI_DRIVE = '/content/drive/MyDrive/omniparser-weights'

def fetch(repo_id, local_dir, drive_dir, sentinel='config.json'):
    if os.path.exists(os.path.join(drive_dir, sentinel)):
        print(f'{repo_id}: already at {drive_dir}, skipping.')
        return
    print(f'{repo_id}: downloading to {local_dir}...')
    snapshot_download(repo_id=repo_id, local_dir=local_dir,
                      resume_download=True, token=os.environ['HF_TOKEN'])
    print(f'{repo_id}: copying to {drive_dir}...')
    shutil.copytree(local_dir, drive_dir, dirs_exist_ok=True)
    print(f'{repo_id}: done.')

fetch('xlangai/Jedi-7B-1080p', JEDI_LOCAL, JEDI_DRIVE)

try:
    fetch('microsoft/OmniParser-v2.0', OMNI_LOCAL, OMNI_DRIVE)
except Exception as e:
    print(f'OmniParser fetch failed (fallback only, non-blocking): {e}')

## Cell 4 — Load JEDI via vLLM

Detect GPU, pick dtype (bf16 on A100/H100, fp16 elsewhere). Colab Pro gives A100 priority but not guarantee.

In [ ]:
import torch
from vllm import LLM, SamplingParams

gpu = torch.cuda.get_device_name(0)
dtype = 'bfloat16' if ('A100' in gpu or 'H100' in gpu) else 'float16'
print(f'GPU: {gpu}')
print(f'dtype: {dtype}')

llm = LLM(
    model='/content/drive/MyDrive/jedi-weights',
    dtype=dtype,
    gpu_memory_utilization=0.9,
    max_model_len=8192,
    limit_mm_per_prompt={'image': 1},
)
print('JEDI loaded.')

## Cell 5 — Smoke query (imports `grounding/` module)

Set `SCREENSHOT_PATH` and `ISSUE_TYPE`, run.

JEDI needs a concrete element description. Vague targets like "the primary CTA button" trigger chat mode instead of grounding. Always name the element by its visible label (e.g., `"a button labeled 'Upload Photo'"`).

This cell imports from `grounding/jedi.py` and `grounding/prompts.py` rather than inlining inference code. The `llm` object from cell 4 is swapped into a `JEDI` instance so we reuse the already-loaded model without paying another 30s load cost.

In [ ]:
import sys
if '/content/GUI-grounded-gen' not in sys.path:
    sys.path.insert(0, '/content/GUI-grounded-gen')

from grounding.jedi import JEDI
from grounding.prompts import CLICK_ELEMENT, format_query

# Reuse the already-loaded llm from cell 4 (skips 30s reload)
jedi = JEDI.__new__(JEDI)
jedi.llm = llm

SCREENSHOT_PATH = '/content/drive/MyDrive/designbench-samples/1.png'
ISSUE_TYPE = "a button labeled 'Upload Photo'"

instruction = format_query(CLICK_ELEMENT, issue_type=ISSUE_TYPE)
result = jedi.query(SCREENSHOT_PATH, instruction)

print(f"Raw:           {result['raw_output']}")
print(f"Original size: {result['original_size']}")
print(f"Resized size:  {result['resized_size']}")
print(f"Point (orig):  {result['point']}")
print(f"Parsed OK:     {result['parse_success']}")

## Cell 6 — Visualize click on original image

Uses `result['point']` from cell 5 (already in original pixel space — the rescale happens inside `JEDI.query`). Draws a red dot, saves to Drive.

In [ ]:
import os
from PIL import Image, ImageDraw

img = Image.open(SCREENSHOT_PATH).convert('RGB')
viz = img.copy()
draw = ImageDraw.Draw(viz)

if result['point']:
    x, y = result['point']
    r = 14
    draw.ellipse((x - r, y - r, x + r, y + r),
                 fill='red', outline='white', width=3)
    print(f'Click at ({x}, {y})')
else:
    print(f'Parse failed: {result["raw_output"]}')

out_dir = '/content/drive/MyDrive/jedi-smoke-out'
os.makedirs(out_dir, exist_ok=True)
viz.save(f'{out_dir}/smoke.png')
print(f'Saved: {out_dir}/smoke.png')
viz

In [ ]:
SYS_INSPECTOR = "You are a UI quality inspector. You identify visually broken elements on a webpage and click on them. Output only: pyautogui.click(x=<int>, y=<int>)"

VARIANTS = [
    # V1: natural paraphrase of the defect
    ("natural-paraphrase",
     None,
     "Click on the element that is overlapping with another element. Output only: pyautogui.click(x=<int>, y=<int>)"),

    # V2: system prompt as inspector
    ("sys-inspector",
     SYS_INSPECTOR,
     "Find and click the element with an occlusion problem."),

    # V3: chain-of-thought, describe then click
    ("chain-of-thought",
     None,
     "Look at this webpage. First, describe what looks visually wrong. Then output the click on the broken element. Format: pyautogui.click(x=<int>, y=<int>)"),

    # V4: explicit defect definition
    ("defect-definition",
     None,
     "An 'occlusion' defect means two UI elements visually overlap each other in a way they shouldn't. Click the element causing occlusion on this page. Output only: pyautogui.click(x=<int>, y=<int>)"),

    # V5: action verb 'point at'
    ("point-at",
     None,
     "Point at the broken element on this page. Output only: pyautogui.click(x=<int>, y=<int>)"),

    # V6: concrete HTML-extracted target (benchmark — we know this works)
    ("concrete-baseline",
     None,
     "Click the language tag labeled 'English'. Output only: pyautogui.click(x=<int>, y=<int>)"),
]

for name, sys, user in VARIANTS:
    msgs = []
    if sys:
        msgs.append({'role': 'system', 'content': sys})
    msgs.append({'role': 'user', 'content': [
        {'type': 'image_url', 'image_url': {'url': f"data:image/png;base64,{__import__('base64').b64encode(open(SCREENSHOT_PATH,'rb').read()).decode()}"}},
        {'type': 'text', 'text': user},
    ]})
    out = llm.chat(msgs, sampling_params=SamplingParams(temperature=0.0, max_tokens=256))
    print(f'\n=== {name} ===')
    print(out[0].outputs[0].text)


In [ ]:
from PIL import Image, ImageDraw
img = Image.open(SCREENSHOT_PATH).convert('RGB')
viz = img.copy()
d = ImageDraw.Draw(viz)
# Draw y=358 horizontal line across image for reference
d.line([(0, 358), (img.size[0], 358)], fill='yellow', width=2)
# Draw the three "defect-aware" clicks
for (x, y, label) in [(601, 358, 'V1'), (581, 358, 'V2'), (685, 358, 'V4'), (520, 507, 'V6-baseline')]:
    d.ellipse((x-12, y-12, x+12, y+12), outline='red', width=3)
    d.text((x+15, y-8), label, fill='red')
viz


## What worked / how to use

### Canonical code lives in `grounding/`
- [`grounding/jedi.py`](../grounding/jedi.py) — `JEDI` class with `query(image_path, instruction) -> dict`. Lazy-imports torch/vllm so it's safe to import on Mac (no GPU) for batch.py development.
- [`grounding/prompts.py`](../grounding/prompts.py) — `CLICK_ELEMENT` template + `format_query()` helper.

**This notebook is the reference implementation / smoke harness.** Production code (batch runs, A/B, pipeline integration) imports from `grounding/`.

### Config that works
- **Runtime:** Colab Pro A100-SXM4-40GB, vLLM bf16
- **Model:** `xlangai/Jedi-7B-1080p`, weights cached at `/content/drive/MyDrive/jedi-weights/`
- **Prompt pattern:** `"Click the element with {issue_type}. Output only: pyautogui.click(x=<int>, y=<int>)"`
- **Output format JEDI returns:** `x=<float> y=<float>` (resized-image space)
- **Coord rescale:** `orig_x = cx_resized * orig_w / resized_w`
- **Image preprocess:** `smart_resize(h, w, factor=28, min_pixels=256*28*28, max_pixels=1280*28*28)`, then base64 data URL via `llm.chat` with `image_url`

### What fails silently
- **Vague targets** ("the primary CTA button") → JEDI falls back to chat mode, replies in natural language. Always name element by visible label.
- **`userdata.get('HF_TOKEN')`** in VS Code Colab kernel → timeout. Use `getpass` (cell 2).
- **`{'type': 'image', 'image': PIL.Image}`** in vLLM chat API → `NotImplementedError: Unknown part type: image`. Must use `image_url` with base64 data URL.
- **`smart_resize(h, w, min_pixels=..., max_pixels=...)`** without `factor=28` → `TypeError: missing required positional argument 'factor'`.

### How to run
1. Open notebook in VS Code with the **Google Colab** extension, connect to A100 runtime
2. Run cells 1–4 in order: bootstrap (includes `git pull`) → HF token → weight download → JEDI load
3. In cell 5: set `SCREENSHOT_PATH` and `ISSUE_TYPE`, run
4. Cell 6 draws red dot on original image, saves to `/content/drive/MyDrive/jedi-smoke-out/smoke.png`

### Day 1 handoff
- **C2 (batch):** `grounding/batch.py` imports `JEDI` + `CLICK_ELEMENT`, loops over DesignBench oracle issues per sample, writes `grounding/cache/{fw}_{i}.json` per the schema in `jedi_integration_plan.md`.
- **C3 (prompt A/B):** 3 Qwen72B repair-prompt variants live in `pipeline/prompts.py`.
- **C4 (pipeline):** `pipeline/run.py` loads grounding cache + calls Qwen72B.

## Cell 8 — OmniParser prod test (v1 + v2 + structural)

Runs all three OmniParser variants on 5 DesignBench samples (one per defect type). Saves JSON + annotated PNGs to `/content/drive/MyDrive/omniparser-test/`.

- **A:** Pin `transformers==4.49.0` + `vllm==0.8.2` (one-time per runtime). **Restart runtime** after this cell, then re-run cells 1-4.
- **B:** Pull latest code and run the prod script (~5-8 min on A100).

Requires OmniParser weights at `/content/drive/MyDrive/omniparser-weights/` (already cached from cell 3).

**Why the pins:** Florence-2 (inside OmniParser) was written for transformers 4.49 and Microsoft never updated it. vllm 0.8.2 is the highest that accepts transformers 4.49 AND has Qwen2.5-VL (for JEDI). With these pins, both run in the same kernel — no patches needed.

In [ ]:
# Pin to versions that work for BOTH JEDI (vllm) AND OmniParser (Florence-2).
# Required because Microsoft's Florence-2 trust_remote_code was written for
# transformers 4.49; 4.50+ breaks it. vllm 0.8.2 is the highest that still
# accepts transformers 4.49 AND supports Qwen2.5-VL.
# See plans/giggly-snuggling-wand.md for investigation.
#
# Run ONCE, then restart runtime so pip changes take effect.
!pip install -q 'transformers==4.49.0' 'vllm==0.8.2' easyocr safetensors ultralytics --force-reinstall
print("Pip install done. RESTART RUNTIME now, then re-run cells 1-4.")

In [ ]:
!cd /content/GUI-grounded-gen && git pull
!python /content/GUI-grounded-gen/scripts/prod_omniparser.py